In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                              roc_curve, roc_auc_score, RocCurveDisplay,
                              precision_recall_curve, average_precision_score)


In [ ]:
df = pd.read_excel('Practice_Dataset.xlsx')

clf_features = ['punch_count', 'hours_worked', 'satisfaction_score']
df_clf = df[clf_features + ['is_absent']].copy()

# Same fix as Day 3: hours_worked is NaN exactly when the employee is absent,
# so fill with 0 instead of dropping those rows.
df_clf['hours_worked'] = df_clf['hours_worked'].fillna(0)
df_clf['satisfaction_score'] = df_clf['satisfaction_score'].fillna(df_clf['satisfaction_score'].median())

Xc = df_clf[clf_features]
yc = df_clf['is_absent']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=42, stratify=yc
)

param_grid_clf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [4, 8, None],
    'class_weight': [None, 'balanced'],
}
grid_clf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_clf, cv=5, scoring='f1', n_jobs=-1
)
grid_clf.fit(Xc_train, yc_train)

# Task 4.1 — Confusion Matrix Deep Dive

In [ ]:
best_clf = grid_clf.best_estimator_
y_pred = best_clf.predict(Xc_test)
cm = confusion_matrix(yc_test, y_pred)
print('Confusion Matrix:')
print(cm)
print()
print('              Predicted: No  Predicted: Yes')
print(f'Actual: No       {cm[0][0]:>4}          {cm[0][1]:>4}')
print(f'Actual: Yes      {cm[1][0]:>4}          {cm[1][1]:>4}')

print('\nTrue Negatives: ', cm[0][0], ' --- correctly predicted NOT absent')
print('False Positives:', cm[0][1], ' --- predicted absent, was actually present')
print('False Negatives:', cm[1][0], ' --- predicted present, was actually absent (risky miss!)')
print('True Positives: ', cm[1][1], ' --- correctly predicted absent')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=['Present', 'Absent']).plot(ax=ax, cmap='Blues')
plt.savefig('week5/confusion_matrix_detailed.png', dpi=150)
plt.show()

# Task 4.2 — ROC Curve & AUC

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

# ROC curve shows the tradeoff between True Positive Rate and False Positive Rate
# at every possible classification threshold
y_proba = best_clf.predict_proba(Xc_test)[:, 1]  # probability of class 1
fpr, tpr, thresholds = roc_curve(yc_test, y_proba)
auc = roc_auc_score(yc_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#0D9488', label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='#94A3B8', linestyle='--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — is_absent Classifier')
plt.legend()
plt.tight_layout()
plt.savefig('week5/roc_curve.png', dpi=150)
plt.show()
print(f'AUC: {auc:.3f}')
# AUC = 0.5 -> random guessing, AUC = 1.0 -> perfect classifier